In [1]:
# General notebook settings
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
# pandas<3.0.3 sets the `locs` attribute deprecated in matplotlib>=3.11
warnings.filterwarnings("ignore", message="The locs attribute was deprecated")
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Backpressure CHP

This example demonstrates how to model a Combined Heat and Power (CHP) plant with a fixed heat-power ratio, assuming that the plant is operated in backpressure mode.
For an example of a CHP plant with a more complicated heat-power feasible operational area, see the [extraction-condensing CHP example](./power-to-gas-boiler-chp.ipynb). In this example, the CHP is modelled as a `Link` component with two output buses: one for electricity and one for heat. The CHP unit must be heat-following since there is no other supply of heat to the "Frankfurt heat" bus.

In [2]:
import pypsa

n = pypsa.Network()

n.add("Bus", "Frankfurt", carrier="AC")
n.add("Load", "Frankfurt", bus="Frankfurt", p_set=5)

n.add("Bus", "Frankfurt heat", carrier="heat")
n.add("Load", "Frankfurt heat", bus="Frankfurt heat", p_set=3)

n.add("Bus", "Frankfurt gas", carrier="gas")
n.add("Generator", "Frankfurt gas", bus="Frankfurt gas", marginal_cost=100, p_nom=100)

n.add(
    "Link",
    "OCGT",
    bus0="Frankfurt gas",
    bus1="Frankfurt",
    p_nom_extendable=True,
    capital_cost=600,
    efficiency=0.4,  # electricity per unit of gas
)

n.add(
    "Link",
    "CHP",
    bus0="Frankfurt gas",
    bus1="Frankfurt",
    bus2="Frankfurt heat",
    p_nom_extendable=True,
    capital_cost=1400,
    efficiency=0.3,  # electricity per unit of gas
    efficiency2=0.3,  # heat per unit of gas
)

n.optimize();

/home/runner/work/PyPSA/PyPSA/pypsa/network/io.py:2082: FutureWarning: pandas infers the `str` dtype for string data since its version 3.0. PyPSA still converts it back to numpy object dtype on import, but will keep it from PyPSA 2.0 on. Set `pypsa.options.api.legacy_string_dtype` explicitly to suppress this warning.
  new_static = _coerce_string_dtypes(new_static)


/tmp/ipykernel_2862/1890019209.py:36: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n.optimize();
Index(['Frankfurt', 'Frankfurt heat', 'Frankfurt gas'], dtype='object', name='name')


Index(['OCGT', 'CHP'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


...

Reading time = 0.00 seconds


obj: 11 rows, 5 columns, 16 nonzeros


Set parameter LogToConsole to value 0


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 5 primals, 11 duals
Objective: 1.85e+04
Solver: gurobi
Runtime: 0.00s
Dual bound: 1.85e+04
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-ext-p-lower, Link-ext-p-upper were not assigned to the network.


In [3]:
n.loads_t.p

name,Frankfurt,Frankfurt heat
snapshot,,
now,5.0,3.0


In [4]:
n.links_t.p0

name,OCGT,CHP
snapshot,,
now,5.0,10.0


In [5]:
n.links_t.p1

name,OCGT,CHP
snapshot,,
now,-2.0,-3.0


In [6]:
n.links_t.p2

name,OCGT,CHP
snapshot,,
now,0.0,-3.0
